In [ ]:
!pip install langchain langchain-core langchain-community langchain-groq faiss-cpu

In [2]:
from langchain_groq import ChatGroq

from dotenv import load_dotenv
load_dotenv()

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

response = llm.invoke("What is the best phone under 15000?")
print(response.content)

The best phone under 15000 can vary depending on personal preferences, needs, and the latest market trends. However, here are some top options to consider:

1. **Redmi 12**: This phone offers a 6.7-inch display, 50MP primary camera, 5000mAh battery, and up to 8GB of RAM. It's available for around 12,000-14,000 INR.
2. **Samsung Galaxy M33**: This phone features a 6.6-inch display, 50MP primary camera, 5000mAh battery, and up to 8GB of RAM. It's priced around 13,000-15,000 INR.
3. **Realme 9 Pro**: This phone boasts a 6.6-inch display, 48MP primary camera, 5000mAh battery, and up to 8GB of RAM. It's available for around 12,000-14,000 INR.
4. **Xiaomi Poco X4 Pro**: This phone offers a 6.67-inch display, 48MP primary camera, 5000mAh battery, and up to 8GB of RAM. It's priced around 13,000-15,000 INR.
5. **Nokia G60**: This phone features a 6.58-inch display, 50MP primary camera, 4500mAh battery, and up to 6GB of RAM. It's available for around 12,000-14,000 INR.

When choosing the best ph

In [4]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

pompt  = PromptTemplate(
    template = "what is the capital of the country: {country}",
    input_variables=["country"],
)
output_parser = StrOutputParser()
chain = pompt | llm | output_parser
response = chain.invoke({"country": "India"})
print(response)

The capital of India is New Delhi.


In [10]:
from langchain_core.runnables import RunnableParallel
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt1 = ChatPromptTemplate.from_template("What is the capital of {country}?")
prompt2 = ChatPromptTemplate.from_template("What is the population of {country}?")     
prompt3 = ChatPromptTemplate.from_template("combine the above information and summarize {capital} and {population}")

output_parser = StrOutputParser()

temp1 = RunnableParallel(
    {
    "capital": prompt1 | llm | output_parser,
    "population": prompt2 | llm | output_parser
    }
)

chain = temp1 | prompt3 | llm | output_parser


print(chain.invoke({"country": "India"}))


Here's a summary of the information:

The capital of India is New Delhi. As of my cut-off knowledge in 2023, the estimated population of India is approximately 1.417 billion people. Please note that this information may have changed since my knowledge cutoff date, and for the most up-to-date information, I recommend checking with a reliable source such as the World Bank or the United Nations.


In [17]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser   

run = RunnableParallel(
    original_input = RunnablePassthrough(),
  
)

run.invoke({"num": 5})

{'original_input': {'num': 5}}

In [19]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser   

run = RunnableParallel(
    original_input = RunnablePassthrough(),
    modified_input = lambda x: {"num": x["num"] + 3},
    doubled = lambda x: x["num"] * 2
  
)

run.invoke({"num": 5})

{'original_input': {'num': 5}, 'modified_input': {'num': 8}, 'doubled': 10}

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt1 = ChatPromptTemplate.from_template("What is the capital of {country}?")
prompt2 = ChatPromptTemplate.from_template("What is the population of {country}?")     
prompt3 = ChatPromptTemplate.from_template("combine the above information and summarize {capital} and {population}")

output_parser = StrOutputParser()

chain1 = RunnableParallel(
    {
    "capital": prompt1 | llm | output_parser,
    "population": prompt2 | llm | output_parser
    }
)

chain2 = RunnableParallel(
    {
        "intermediate" : RunnablePassthrough(),
        "summary": prompt3 | llm | output_parser
    }
  
)

chain = chain1 | chain2


result =chain.invoke({"country": "India"})

print(result)


{'intermediate': {'capital': 'The capital of India is New Delhi.', 'population': 'As of my cut-off knowledge in 2023, the estimated population of India is approximately 1.417 billion people. However, please note that this information may have changed since my knowledge cutoff date. For the most up-to-date information, I recommend checking with a reliable source such as the World Bank or the United Nations.'}, 'summary': "Here's a summary of the information:\n\nThe capital of India is New Delhi. As of my cut-off knowledge in 2023, the estimated population of India is approximately 1.417 billion people. Please note that this information may have changed since my knowledge cutoff date, and for the most up-to-date information, I recommend checking with a reliable source such as the World Bank or the United Nations."}


In [ ]:
result['intermediate']['population']

'As of my cut-off knowledge in 2023, the estimated population of India is approximately 1.417 billion people. However, please note that this information may have changed since my knowledge cutoff date. For the most up-to-date information, I recommend checking with a reliable source such as the World Bank or the United Nations.'

In [17]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


# Step 1: Clean text
def clean_text(data: dict) -> dict:
    data["cleaned_text"] = data["email"].strip().lower()
    return data

# Step 2: Extract metadata
def extract_info(data: dict) -> dict:
    data["num_words"] = len(data["cleaned_text"].split())
    return data

# Step 3: Prepare final output
def format_output(data: dict) -> dict:
    return {
        "original_email": data["email"],
        "cleaned_text": data["cleaned_text"],
        "num_words": data["num_words"],
        "sentiment": data["sentiment"]
    }

clean = RunnableLambda(clean_text)
metadata_extractor = RunnableLambda(extract_info)
formatter = RunnableLambda(format_output)

# Prompt
prompt = ChatPromptTemplate.from_template(
    "Analyze the sentiment of this email: {cleaned_text}. "
    "Return only one word: positive, negative, or neutral."
)

output_parser = StrOutputParser()

# Step 4: Sentiment branch
sentiment_chain = prompt | llm | output_parser

# Step 5: Combine everything
chain = (
    RunnablePassthrough()
    | clean
    | metadata_extractor
    | RunnableParallel({
        "email": lambda x: x["email"],
        "cleaned_text": lambda x: x["cleaned_text"],
        "num_words": lambda x: x["num_words"],
        "sentiment": sentiment_chain
    })
    | formatter
)

# Input
test_email = {"email": "Hello, I am very happy with my purchase!"}

# Run
result = chain.invoke(test_email)

print(result)

{'original_email': 'Hello, I am very happy with my purchase!', 'cleaned_text': 'hello, i am very happy with my purchase!', 'num_words': 8, 'sentiment': 'Positive'}
